
# 🔁 Daily Repo Loop for **Ego4d-LiteSTA** (Colab-first)

This block sets up SSH to GitHub (**Deploy Key stored in Drive**), pulls your repo, runs code, logs outputs, and pushes logs/artifacts back to GitHub.
- One-time: add your **Deploy Key** public key to your GitHub repo Deploy Keys with **write access**.
- Edit the variables (**GITHUB_USER**, **REPO_NAME**) below once; they persist per notebook.


In [ ]:
# ⛳ Mount Drive and set variables
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USER = "your-gh-username"          # <<< EDIT ME
REPO_NAME   = "Ego4d-LiteSTA"             # <<< default as requested
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
WORKDIR     = f"/content/{REPO_NAME}"
SSH_DIR     = "/content/drive/MyDrive/.ssh"

print("Repo:", REPO_SSH)
print("Workdir:", WORKDIR)

In [2]:
# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash -s "$SSH_DIR"
SSH_DIR="$1"
mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"
eval "$(ssh-agent -s)"
ssh-add "$SSH_DIR/colab_egolite" >/dev/null 2>&1 || true
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat <<EOF > ~/.ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile $SSH_DIR/colab_egolite
  IdentitiesOnly yes
StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config
ssh-keyscan -t rsa,ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null || true
chmod 644 ~/.ssh/known_hosts

Agent pid 861


## Pull and Push — Daily Sync
Run after setting variables and SSH above.


In [ ]:
%cd "$WORKDIR"
!bash pull.sh


In [ ]:
%cd "$WORKDIR"
!bash push.sh


In [ ]:
# 🧾 Run ID + logger helpers (use these in your train/eval cells)
import os, json, traceback, datetime, shutil, glob

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = "logs"; RUN_DIR = f"runs/{RUN_ID}"; OUT_DIR = "outputs"
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

LOG_PATH = f"{LOG_DIR}/{RUN_ID}.log"
ERR_FILE = "COLAB_ERRORS.md"

def log_write(msg):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def record_error(title, err_text):
    with open(ERR_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n## {RUN_ID} — {title}\n\n```\n{err_text}\n```\n")

log_write(f"New run: {RUN_ID}")

# 🚀 Ego4D STA — Full Colab (0→100)

This notebook sets up a lightweight end-to-end baseline for the **Short-Term Object Interaction Anticipation (STA)** task on Ego4D.

**What you get:**
- Setup (GPU check, deps, Drive)
- Paths & data checks
- Load STA annotations & visualize a sample
- Minimal datasets (frames or features)
- Lightweight model (MViTv2-S + tiny heads + YOLOv8n proposals)
- Short training loop
- Export predictions JSON for official eval

**Tip:** Keep runs short on Colab Free and log outputs to Drive.

In [ ]:
# 🔍 Runtime & GPU
import torch, platform, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))


In [ ]:
# 🔗 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# 📦 Deps (keep minimal)
!pip -q install timm==0.9.16 opencv-python==4.10.0.84 matplotlib==3.9.2 ultralytics==8.3.36


In [ ]:
# 📁 Paths & mode
from pathlib import Path

EGO4D_DIR = Path('/content/drive/MyDrive/ego4d_data')  # <-- edit if needed
SPLIT = 'v2'
MODE = 'frames'  # 'frames' or 'features'

ANN_DIR = EGO4D_DIR / SPLIT / 'annotations'
FRAMES_DIR = EGO4D_DIR / SPLIT / 'frames'
STA_TRAIN_JSON = ANN_DIR / 'fho_sta_train.json'
STA_VAL_JSON   = ANN_DIR / 'fho_sta_val.json'
META_JSON      = EGO4D_DIR / SPLIT / 'ego4d.json'

print('EGO4D_DIR :', EGO4D_DIR)
print('STA train :', STA_TRAIN_JSON, STA_TRAIN_JSON.exists())
print('STA val   :', STA_VAL_JSON, STA_VAL_JSON.exists())
print('Frames    :', FRAMES_DIR, FRAMES_DIR.exists())


In [ ]:
# 📚 Load annotations & pick a sample
import json, random

def read_json(p):
    with open(p, 'r') as f:
        return json.load(f)

sta_val = read_json(STA_VAL_JSON)
sta_train = read_json(STA_TRAIN_JSON) if STA_TRAIN_JSON.exists() else {'annotations': []}

print('Train anns:', len(sta_train.get('annotations', [])))
print('Val anns  :', len(sta_val.get('annotations', [])))

# Pick a sample with at least one object
SAMPLE = None
for a in sta_val.get('annotations', []):
    if a.get('objects'):
        SAMPLE = a
        break
print('Sample clip_uid:', SAMPLE.get('clip_uid') if SAMPLE else None)


In [ ]:
# 👀 Visualize last frame + one GT box (if frames exist)
import os, glob, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (5,5)

def last_frame_path(video_uid):
    d = FRAMES_DIR / video_uid
    imgs = sorted(glob.glob(str(d/'*.jpg'))) or sorted(glob.glob(str(d/'*.png')))
    return imgs[-1] if imgs else None

if SAMPLE is not None:
    vid = SAMPLE.get('video_id') or SAMPLE.get('video_uid')
    lf = last_frame_path(vid)
    print('video_uid:', vid)
    print('last frame exists:', os.path.exists(lf) if lf else None)
    if lf and os.path.exists(lf):
        img = cv2.imread(lf)[:,:,::-1]
        objs = SAMPLE.get('objects', [])
        if objs and objs[0].get('box'):
            x1,y1,x2,y2 = map(int, objs[0]['box'])
            cv2.rectangle(img, (x1,y1), (x2,y2), (255,255,255), 3)
        plt.axis('off'); plt.imshow(img); plt.show()
else:
    print('No sample to visualize.')


In [ ]:
# 🧱 Datasets
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import List, Optional, Dict, Any
import cv2

@dataclass
class STAObject:
    bbox: List[float]
    verb_id: Optional[int]
    noun_id: Optional[int]
    ttc: Optional[float]

@dataclass
class Sample:
    clip_uid: str
    video_uid: str
    objects: List[STAObject]
    last_frame: Optional[str] = None

def build_index(sta_dict: Dict[str,Any]):
    out = []
    for a in sta_dict.get('annotations', []):
        vid = a.get('video_id') or a.get('video_uid')
        if not vid: continue
        objs = [STAObject(o.get('box'), o.get('verb_category_id'), o.get('noun_category_id'), o.get('time_to_contact')) for o in a.get('objects', [])]
        d = FRAMES_DIR / vid
        imgs = sorted(list(d.glob('*.jpg'))) or sorted(list(d.glob('*.png')))
        last = str(imgs[-1]) if imgs else None
        out.append(Sample(a.get('clip_uid'), vid, objs, last))
    return out

class FramesDataset(Dataset):
    def __init__(self, sta_dict, clip_len=12, resize_short=192):
        self.idx = build_index(sta_dict)
        self.clip_len = clip_len
        self.resize_short = resize_short
    def __len__(self): return len(self.idx)
    def _load_clip(self, last_path):
        folder = Path(last_path).parent
        imgs = sorted(list(folder.glob('*.jpg'))) or sorted(list(folder.glob('*.png')))
        take = imgs[-self.clip_len:]
        frames = []
        for p in take:
            im = cv2.imread(str(p))
            if im is None: continue
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
            h,w = im.shape[:2]
            scale = self.resize_short / min(h,w)
            im = cv2.resize(im, (int(w*scale), int(h*scale)))
            frames.append(im)
        if not frames:
            return None
        H = min(f.shape[0] for f in frames)
        W = min(f.shape[1] for f in frames)
        frames = [f[:H,:W,:] for f in frames]
        arr = np.stack(frames).transpose(0,3,1,2)/255.0
        return torch.from_numpy(arr).float()
    def __getitem__(self, i):
        s = self.idx[i]
        vid = None
        if s.last_frame:
            vid = self._load_clip(s.last_frame)
        return {'clip_uid': s.clip_uid, 'video_uid': s.video_uid, 'video': vid, 'last_frame': s.last_frame, 'objects': s.objects}

BATCH=4
train_ds = FramesDataset(sta_train)
val_ds   = FramesDataset(sta_val)

def collate(batch):
    out = {}
    for k in batch[0].keys():
        out[k] = [b[k] for b in batch]
    return out

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, collate_fn=collate)
print('train/val sizes:', len(train_ds), len(val_ds))


In [ ]:
# 🧠 Model (MViTv2-S + tiny heads + YOLOv8n proposals)
import torch.nn as nn
from torchvision.models.video import mvit_v2_s, MViT_V2_S_Weights
from ultralytics import YOLO

class Selector(nn.Module):
    def __init__(self, d, h=512):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,h), nn.ReLU(True), nn.Linear(h,1))
    def forward(self, x):
        B,K,D = x.shape
        y = self.net(x.reshape(B*K,D)).reshape(B,K)
        return y

class VerbHead(nn.Module):
    def __init__(self, d, nverb=120):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d,d//2), nn.ReLU(True), nn.Linear(d//2,nverb))
    def forward(self, x): return self.fc(x)

class TTCHead(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d,d//2), nn.ReLU(True), nn.Linear(d//2,1))
    def forward(self, x): return self.fc(x).squeeze(-1)

class STALite(nn.Module):
    def __init__(self, nverb=120):
        super().__init__()
        self.backbone = mvit_v2_s(weights=MViT_V2_S_Weights.KINETICS400_V1)
        self.backbone.classifier[-1] = nn.Identity()
        self.d = 768
        self.selector = Selector(self.d)
        self.verb = VerbHead(self.d, nverb)
        self.ttc  = TTCHead(self.d)
        self.det  = YOLO('yolov8n.pt')
    def encode(self, vids):
        # vids: list of TCHW
        T = max(v.shape[0] for v in vids if v is not None) if any(v is not None for v in vids) else 8
        if T<1: T=8
        batch = []
        for v in vids:
            if v is None:
                batch.append(torch.zeros(8,3,112,112))
            else:
                H = min(v.shape[2], 224)
                W = min(v.shape[3], 224)
                vv = v[:T,:,:H,:W]
                batch.append(vv)
        x = torch.stack(batch,0).permute(0,2,1,3,4).contiguous()  # B,C,T,H,W
        return self.backbone(x)
    def forward(self, batch, device):
        feats = self.encode([v.to(device) if isinstance(v, torch.Tensor) else None for v in batch['video']])
        B,D = feats.shape
        # candidates from last frame
        KMAX=8
        boxes = []
        scores= []
        for p in batch['last_frame']:
            if p is None:
                boxes.append(torch.zeros(KMAX,4))
                scores.append(torch.zeros(KMAX))
                continue
            r = self.det.predict(source=p, imgsz=448, conf=0.25, max_det=KMAX, verbose=False)[0]
            bx = r.boxes.xyxy.cpu() if (r and r.boxes is not None) else torch.zeros((0,4))
            sc = r.boxes.conf.cpu()  if (r and r.boxes is not None) else torch.zeros((0,))
            K = min(KMAX, bx.shape[0])
            Bpad = torch.zeros(KMAX,4); Spad = torch.zeros(KMAX)
            if K>0:
                Bpad[:K]=bx[:K]; Spad[:K]=sc[:K]
            boxes.append(Bpad); scores.append(Spad)
        boxes = torch.stack(boxes,0).to(feats.device)
        scores= torch.stack(scores,0).to(feats.device)
        # score candidates using clip features (broadcast)
        featsK = feats.unsqueeze(1).expand(-1, boxes.shape[1], -1)
        sel_logits = self.selector(featsK)
        best = (sel_logits + scores).argmax(dim=1)
        verb_logits = self.verb(feats)
        ttc = self.ttc(feats)
        return {'boxes': boxes, 'sel_logits': sel_logits, 'best': best, 'verb_logits': verb_logits, 'ttc': ttc}

model = STALite()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print('Model ready on', device)


In [ ]:
# 🏋️ Train (very small loop)
import torch.nn.functional as F

def iou_xyxy(a,b):
    if a.numel()==0 or b.numel()==0:
        return torch.zeros(a.shape[0], b.shape[0])
    tl = torch.max(a[:,None,:2], b[:,:2])
    br = torch.min(a[:,None,2:], b[:,2:])
    wh = (br-tl).clamp(min=0)
    inter = wh[:,:,0]*wh[:,:,1]
    area_a = (a[:,2]-a[:,0]).clamp(min=0)*(a[:,3]-a[:,1]).clamp(min=0)
    area_b = (b[:,2]-b[:,0]).clamp(min=0)*(b[:,3]-b[:,1]).clamp(min=0)
    return inter/(area_a[:,None]+area_b-inter+1e-6)

def build_targets(batch, pred_boxes, device):
    B,K,_ = pred_boxes.shape
    sel_t = torch.zeros(B, dtype=torch.long, device=device)
    verb_t= torch.zeros(B, dtype=torch.long, device=device)
    ttc_t = torch.zeros(B, dtype=torch.float32, device=device)
    has   = torch.zeros(B, dtype=torch.bool, device=device)
    for i in range(B):
        objs = batch['objects'][i]
        if not objs: continue
        gt = None
        for o in objs:
            if o.bbox:
                gt = torch.tensor(o.bbox, device=device).float()
                verb_t[i] = int(o.verb_id) if isinstance(o.verb_id,int) else 0
                ttc_t[i]  = float(max(0.0, min(5.0, o.ttc if o.ttc is not None else 0.0)))
                break
        if gt is None: continue
        has[i]=True
        ious = iou_xyxy(pred_boxes[i], gt.unsqueeze(0)).squeeze(1)
        sel_t[i] = int(torch.argmax(ious).item())
    return sel_t, verb_t, ttc_t, has

opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=5e-2)
EPOCHS=1

for e in range(EPOCHS):
    model.train()
    for step, batch in enumerate(train_loader):
        out = model(batch, device)
        sel_t, verb_t, ttc_t, has = build_targets(batch, out['boxes'], device)
        if has.any():
            loss_sel = F.cross_entropy(out['sel_logits'][has], sel_t[has])
            loss_verb= F.cross_entropy(out['verb_logits'][has], verb_t[has])
            loss_ttc = F.smooth_l1_loss(out['ttc'][has], ttc_t[has])
            loss = loss_sel + loss_verb + 0.5*loss_ttc
        else:
            loss = out['sel_logits'].sum()*0
        opt.zero_grad(); loss.backward(); opt.step()
        if step%20==0:
            print(f'epoch {e} step {step} loss {loss.item():.4f}')
    print('epoch', e, 'done')

# quick val stats
model.eval()
correct=0; total=0
with torch.no_grad():
    for batch in val_loader:
        out = model(batch, device)
        sel_t, verb_t, ttc_t, has = build_targets(batch, out['boxes'], device)
        if has.any():
            pred = out['best'][has]
            correct += (pred==sel_t[has]).sum().item()
            total   += has.sum().item()
print('Selector acc (val, gt match proxy):', correct/max(1,total))


In [ ]:
# 📄 Export predictions JSON (val)
import json

preds = {}
model.eval()
with torch.no_grad():
    for batch in val_loader:
        out = model(batch, device)
        B = len(batch['clip_uid'])
        for i in range(B):
            cu = batch['clip_uid'][i]
            k  = int(out['best'][i].item())
            box = out['boxes'][i,k].detach().cpu().tolist()
            score = float(torch.softmax(out['sel_logits'][i], dim=0)[k].item())
            verb  = int(torch.argmax(out['verb_logits'][i]).item())
            noun  = -1
            ttc   = float(max(0.0, min(5.0, out['ttc'][i].item())))
            preds.setdefault(cu, []).append({
                'box':[round(x,2) for x in box],
                'noun_category_id': noun,
                'verb_category_id': verb,
                'time_to_contact': round(ttc,2),
                'score': round(score,4)
            })

OUT_JSON = '/content/sta_val_preds.json'
with open(OUT_JSON,'w') as f:
    json.dump({'results': preds}, f)
print('Wrote', OUT_JSON)
